In [2]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


In [3]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Writing delivery/data.py


In [4]:
%%writefile delivery/features.py

def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y


# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Writing delivery/features.py


In [5]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Writing delivery/model.py


In [6]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Writing delivery/validate.py


In [7]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Writing delivery/__init__.py


In [8]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.4


In [9]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Writing train.py


In [10]:
!python train.py

Test MAE: 1.4
Training done


In [11]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Writing predict.py


In [12]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 37.9 minutes


In [13]:
!python predict.py -3 15 2 0

Invalid order


In [14]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


In [16]:
import os
print(os.getcwd())

c:\Users\Maanvi\Desktop\Mlop's_Lab


In [ ]:
import os
print(os.listdir())

['.anaconda', '.antigravity', '.aws', '.cache', '.conda', '.condarc', '.continuum', '.copilot', '.docker', '.gemini', '.ghcp-appmod', '.ghcp-appmod-java', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.kiro', '.lesshst', '.matplotlib', '.node_repl_history', '.packettracer', '.ssh', '.streamlit', '.vscode', '.vscode-shared', '3D Objects', 'A.java', 'anaconda3', 'AppData', 'Application Data', 'AppMods', 'c++', 'Cisco Packet Tracer 9.0.0', 'Contacts', 'Cookies', 'data', 'delivery', 'delivery_model.joblib', 'Desktop', 'Documents', 'Downloads', 'Favorites', 'Links', 'Local Settings', 'Main.java', 'MLOPS_LAB4.ipynb', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{80053950-e063-11ef-af17-8e7e7a270386}.TM.blf', 'NTUSER.DAT{80053950-e063-11ef-af17-8e7e7a270386}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{80053950-e063-11ef-af17-8e7e7a270386}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'On

In [18]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (455, 30)
Testing samples: (114, 30)


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

models = {
    "LogisticRegression": LogisticRegression(max_iter=10000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42)
}

best_model = None
best_score = 0
best_name = ""

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    score = scores.mean()

    print(name, ":", round(score, 4))

    if score > best_score:
        best_score = score
        best_model = model
        best_name = name

best_model.fit(X_train, y_train)

joblib.dump(best_model, "classification_model.joblib")

print("\nBest model:", best_name)
print("Best CV accuracy:", round(best_score, 4))
print("Model saved.")

LogisticRegression : 0.9495
DecisionTree : 0.9099
RandomForest : 0.9538

Best model: RandomForest
Best CV accuracy: 0.9538
Model saved.


In [20]:
def validate_input(radius_mean, texture_mean, area_mean):
    
    if radius_mean <= 0 or radius_mean > 50:
        return False

    if texture_mean < 0 or texture_mean > 100:
        return False

    if area_mean <= 0 or area_mean > 3000:
        return False

    return True


print(validate_input(14.0, 20.0, 600.0))
print(validate_input(-5.0, 20.0, 600.0))

True
False


In [21]:
import joblib

model = joblib.load("classification_model.joblib")

sample = X_test[0].reshape(1, -1)

prediction = model.predict(sample)[0]
probabilities = model.predict_proba(sample)[0]

confidence = max(probabilities)

if prediction == 0:
    result = "Malignant"
else:
    result = "Benign"

print("Prediction:", result)
print("Confidence:", round(confidence * 100, 2), "%")

Prediction: Malignant
Confidence: 100.0 %


In [22]:
from sklearn.metrics import confusion_matrix, classification_report
import joblib

def show_metrics(model, X_test, y_test):

    predictions = model.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report:")
    print(classification_report(y_test, predictions))


model = joblib.load("classification_model.joblib")

show_metrics(model, X_test, y_test)

Confusion Matrix:
[[39  3]
 [ 2 70]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

